# AI 기본법 조문 + 항 단위 Qdrant

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET
from common.ai_model import get_embedding_model
from common.qdrant import get_qdrant_client
from qdrant_client.models import Distance, PointStruct, VectorParams
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
XML_PATH = PROJECT_ROOT / 'data' / 'ai_basic_law' / 'ai_basic_law.xml'
root = ET.parse(XML_PATH).getroot()

In [ ]:
chunks = []
CONTENT_TAGS = {'조문내용', '항내용', '호내용', '목내용'}

def text_of(element):
    return ' '.join(''.join(element.itertext()).split())

for unit in root.findall('.//조문단위'):
    article_no = unit.findtext('조문번호', '').strip()
    title = unit.findtext('조문제목', '').strip()
    article_parts = []
    for element in unit:
        if element.tag in CONTENT_TAGS:
            value = text_of(element)
            if value and value not in article_parts:
                article_parts.append(value)
    article_content = '\n'.join(article_parts)
    if article_content and title:
        chunks.append({'id': len(chunks)+1, 'chunk_type': 'article', 'article': f'제{article_no}조', 'paragraph': '', 'title': title, 'content': article_content, 'source': XML_PATH.name})

    for paragraph_no, paragraph in enumerate(unit.findall('./항'), start=1):
        parts = []
        for element in paragraph:
            if element.tag in {'항내용', '호내용', '목내용'}:
                value = text_of(element)
                if value and value not in parts:
                    parts.append(value)
        paragraph_content = '\n'.join(parts)
        if paragraph_content:
            chunks.append({'id': len(chunks)+1, 'chunk_type': 'paragraph', 'article': f'제{article_no}조', 'paragraph': f'{paragraph_no}항', 'title': title, 'content': paragraph_content, 'source': XML_PATH.name})

print(f'조문 + 항 청크 수: {len(chunks)}개')
print(chunks[0])

In [ ]:
embedding_client = get_embedding_model()
qdrant_client = get_qdrant_client()
COLLECTION_NAME = 'ai_basic_law_article_plus_paragraph_v1'
vectors = await embedding_client.aembed_documents([x['content'] for x in chunks])
if qdrant_client.collection_exists(COLLECTION_NAME):
    qdrant_client.delete_collection(COLLECTION_NAME)
qdrant_client.create_collection(collection_name=COLLECTION_NAME, vectors_config=VectorParams(size=len(vectors[0]), distance=Distance.COSINE))
points = [PointStruct(id=x['id'], vector=v, payload=x) for x, v in zip(chunks, vectors)]
qdrant_client.upsert(collection_name=COLLECTION_NAME, points=points)
print(f'{len(points)}개 청크 저장 완료')

In [ ]:
query = '고영향 인공지능이란 무엇인가요?'
query_vector = await embedding_client.aembed_query(query)
result = qdrant_client.query_points(collection_name=COLLECTION_NAME, query=query_vector, limit=5)
for point in result.points:
    p = point.payload
    print(f"[{p['chunk_type']}] {p['article']} {p['paragraph']} {p['title']} score={point.score:.4f}")
    print(p['content'])
    print()